<a href="https://colab.research.google.com/github/IrineuBovoJunior398/Pos-em-IA/blob/main/C%C3%B3pia_de_TAREFA_REDES_NEURAIS_ARTIFICIAIS_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

   Conta  Renda  Dívida Classe
0    101   2800     550    bom
1    102   1300     500    mau
2    103   1400      80    bom
3    104    500     200    mau
4    105   1100     270    mau
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Conta   21 non-null     int64 
 1   Renda   21 non-null     int64 
 2   Dívida  21 non-null     int64 
 3   Classe  21 non-null     object
dtypes: int64(3), object(1)
memory usage: 804.0+ bytes
None


KeyError: "None of [Index(['renda', 'divida'], dtype='object')] are in the [columns]"

In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# 1) Descobrir o nome do arquivo dentro do /content
print("Arquivos em /content:", os.listdir("/content"))

# 2) Ajuste AQUI para o nome real do seu arquivo
arquivo_csv = ("/content/bancario.csv")  # MUDE se necessário

df = pd.read_csv(arquivo_csv)

# Padronizar nomes de colunas (remove espaços e padroniza minúsculo)
df.columns = df.columns.str.strip().str.lower()

print(df.head())
print(df.info())
print("Colunas detectadas:", list(df.columns))

# 3) Detectar coluna de dívida com/sem acento
col_renda = "renda"
col_divida = "dívida" if "dívida" in df.columns else "divida"
col_classe = "classe"

# 4) Conferir se as colunas existem
faltando = [c for c in [col_renda, col_divida, col_classe] if c not in df.columns]
if faltando:
    raise KeyError(f"Faltam colunas no CSV: {faltando}. Colunas existentes: {list(df.columns)}")

# 5) Seleção de colunas
X = df[[col_renda, col_divida]].copy()
y = df[col_classe].copy()

# 6) Garantir numérico (se vier com vírgula decimal, troca por ponto)
for c in [col_renda, col_divida]:
    X[c] = (X[c].astype(str)
              .str.replace(",", ".", regex=False)
              .str.strip())
    X[c] = pd.to_numeric(X[c], errors="coerce")

# Checar NaN nas features
if X.isna().any().any():
    raise ValueError("Há valores não numéricos/NaN em renda/divida após conversão. Verifique o CSV.")

# 7) Mapear classe
y_map = y.astype(str).str.strip().str.lower().map({"bom": 1, "mau": 0})
if y_map.isna().any():
    ruins = y[y_map.isna()].unique()
    raise ValueError(f"Valores inesperados em 'classe': {ruins}. Esperado: 'bom'/'mau'.")

y = y_map.astype(int)

print("Distribuição de classes:")
print(y.value_counts())

# 8) Split
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X.values, y.values, test_size=0.30, random_state=42, stratify=y.values
)

# 9) Padronização
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test  = scaler.transform(X_test_raw)

# 10) Bias
def add_bias(X_):
    return np.hstack([np.ones((X_.shape[0], 1)), X_])

X_train_b = add_bias(X_train)
X_test_b  = add_bias(X_test)

# 11) Perceptron
def step(u):
    return 1 if u >= 0 else 0

def predict_one(x, w):
    return step(np.dot(x, w))

def predict(X_, w):
    return np.array([predict_one(xi, w) for xi in X_])

def train_perceptron(X_, y_, lr=0.01, epochs=200):
    w = np.zeros(X_.shape[1])
    for ep in range(epochs):
        errors = 0
        for xi, yi in zip(X_, y_):
            y_hat = predict_one(xi, w)
            e = yi - y_hat
            if e != 0:
                w = w + lr * e * xi
                errors += 1
        if errors == 0:
            print(f"Convergiu na época {ep+1}.")
            break
    return w

# 12) Experimentos
configs = [
    {"lr": 0.1,   "epochs": 50},
    {"lr": 0.01,  "epochs": 200},
    {"lr": 0.001, "epochs": 500},
]

for cfg in configs:
    print("\n" + "="*60)
    print(f"Config: lr={cfg['lr']} | epochs={cfg['epochs']}")
    w = train_perceptron(X_train_b, y_train, lr=cfg["lr"], epochs=cfg["epochs"])
    y_pred = predict(X_test_b, w)
    print("Acurácia:", accuracy_score(y_test, y_pred))
    print("Matriz de confusão:\n", confusion_matrix(y_test, y_pred))
    print("Relatório:\n", classification_report(y_test, y_pred, zero_division=0))

# 13) Testes fora da amostra
w_final = train_perceptron(X_train_b, y_train, lr=0.01, epochs=200)

idx = 0
x_base = X_test[idx].copy()
x_base_raw = scaler.inverse_transform(x_base.reshape(1, -1))[0]

x_new1 = x_base + np.array([0.6, -0.4])
x_new1_raw = scaler.inverse_transform(x_new1.reshape(1, -1))[0]

print("\n--- Teste A: Perturbação ---")
print("Base (renda, divida) aprox:", x_base_raw, "| prev:", predict_one(np.hstack([1, x_base]), w_final), "| real:", y_test[idx])
print("Nova1 (renda, divida) aprox:", x_new1_raw, "| prev:", predict_one(np.hstack([1, x_new1]), w_final))

x_syn2 = np.array([2.0, -1.5])
x_syn3 = np.array([-1.2, 1.8])

print("\n--- Teste B: Sintéticos ---")
print("Syn2 (renda, divida) aprox:", scaler.inverse_transform(x_syn2.reshape(1, -1))[0], "| prev:", predict_one(np.hstack([1, x_syn2]), w_final))
print("Syn3 (renda, divida) aprox:", scaler.inverse_transform(x_syn3.reshape(1, -1))[0], "| prev:", predict_one(np.hstack([1, x_syn3]), w_final))

Arquivos em /content: ['.config', 'bancario.csv', 'sample_data']
   conta  renda  dívida classe
0    101   2800     550    bom
1    102   1300     500    mau
2    103   1400      80    bom
3    104    500     200    mau
4    105   1100     270    mau
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   conta   21 non-null     int64 
 1   renda   21 non-null     int64 
 2   dívida  21 non-null     int64 
 3   classe  21 non-null     object
dtypes: int64(3), object(1)
memory usage: 804.0+ bytes
None
Colunas detectadas: ['conta', 'renda', 'dívida', 'classe']
Distribuição de classes:
classe
1    12
0     9
Name: count, dtype: int64

Config: lr=0.1 | epochs=50
Convergiu na época 7.
Acurácia: 0.5714285714285714
Matriz de confusão:
 [[3 0]
 [3 1]]
Relatório:
               precision    recall  f1-score   support

           0       0.50      1.00      0.67         3
  